# Stage 5b — Classify, calibrate, and write the dataset

Reads `stage5a_commits.jsonl`, classifies every commit with `model_attention_2024.pt`, calibrates
the confidence, checks it against the maintainer labels, and writes `dataset.csv`.

**The logits are checkpointed in `WORK`**, keyed by commit. A completed run resumes past
classification entirely — this notebook's later sections can be re-run in seconds.

*Its outputs were renamed for publication: the printed output below says `dataset_a.csv` and `dataset_a_meta.json`, which the code now writes as `dataset.csv` and `dataset_meta.json`, the names on Zenodo.*

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Inputs

In [1]:
# Must be set before torch touches CUDA -- once a context exists the flag is ignored.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, gc, math, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

MODEL, MAXLEN, STRIDE = "microsoft/codebert-base", 512, None
LABELS = ["adaptive", "corrective", "perfective"]
TEMPERATURE = 2.86                       # Stage 4c
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DRIVE = WORK          # the setup cell's working folder

SRC = require("stage5a_commits.jsonl")
C = pd.read_json(SRC, lines=True)
MODEL_PATH = require("model_attention_2024.pt")
print(f"{len(C):,} commits from {SRC.name}")
print(f"device {DEVICE}")

Mounted at /content/drive
34,609 commits from stage5a_commits.jsonl
device cuda


## 2. Classify

Memory is handled four ways, each explained in the code comments: memory-efficient attention, bf16,
batching by token budget, and a retry path that releases each failed attempt before the next.

In [2]:
class CommitClassifier(nn.Module):
    def __init__(self, pooling, dropout=0.2):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL)
        self.enc.gradient_checkpointing_enable()
        hid = self.enc.config.hidden_size
        self.pooling = pooling
        if pooling == "attention":
            self.score = nn.Linear(hid, 1)
        else:
            layer = nn.TransformerEncoderLayer(hid, nhead=4, dim_feedforward=512,
                                               dropout=dropout, batch_first=True)
            self.tr = nn.TransformerEncoder(layer, num_layers=1)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hid, len(LABELS))

    def forward(self, ids, att, commit_of, piece_of, n_commits):
        h = self.enc(input_ids=ids, attention_mask=att).last_hidden_state[:, 0]

        key = commit_of * 100_000 + piece_of
        uniq, inv = torch.unique(key, return_inverse=True)
        pv = torch.zeros(len(uniq), h.size(1), device=h.device, dtype=h.dtype)
        pv.index_add_(0, inv, h)
        cnt = torch.zeros(len(uniq), device=h.device, dtype=h.dtype)
        cnt.index_add_(0, inv, torch.ones(len(h), device=h.device, dtype=h.dtype))
        pv = pv / cnt.unsqueeze(1)
        pc = (uniq // 100_000).long()

        per = torch.bincount(pc, minlength=n_commits)
        P = int(per.max().item())
        x = torch.zeros(n_commits, P, h.size(1), device=h.device, dtype=h.dtype)
        m = torch.zeros(n_commits, P, device=h.device, dtype=h.dtype)
        slot = torch.zeros(n_commits, dtype=torch.long, device=h.device)
        for i in range(len(pc)):
            c = pc[i]; x[c, slot[c]] = pv[i]; m[c, slot[c]] = 1; slot[c] += 1

        mm = m.unsqueeze(-1)
        if self.pooling == "attention":
            a = self.score(x).masked_fill(mm == 0, -1e9).softmax(1)
            v = (x * a).sum(1)
        else:
            hh = self.tr(x, src_key_padding_mask=(m == 0))
            v = (hh * mm).sum(1) / mm.sum(1).clamp(min=1)
        return self.out(self.drop(v))


# ---------------------------------------------------------------------------------------------
# Memory-efficient attention.
#
# The default attention implementation materialises a 12 x 512 x 512 score matrix per sequence,
# then a second one after the softmax -- 25 MB per sequence, 64% of everything the encoder
# allocates, and quadratic in sequence length. That is what has been running out of memory, not
# the batch size.
#
# scaled_dot_product_attention computes the same result in tiles and never holds the full
# matrix. It is exact, not an approximation, and PyTorch dispatches to FlashAttention or the
# memory-efficient kernel automatically.
#
# bfloat16 halves what remains. It costs nothing in fidelity here: Stage 4b trained this model
# under torch.autocast(bfloat16), so this is the precision the weights were fitted at.
# ---------------------------------------------------------------------------------------------


tok = AutoTokenizer.from_pretrained(MODEL)

try:
    enc_kwargs = {"attn_implementation": "sdpa"}
    model = CommitClassifier("attention")
    model.enc = AutoModel.from_pretrained(MODEL, **enc_kwargs)
    ATTN = "sdpa"
except Exception as e:
    model = CommitClassifier("attention")
    ATTN = "eager"
    print(f"   SDPA unavailable ({type(e).__name__}); falling back to eager attention.")
    print(f"   Expect roughly 3x the memory per sequence.")

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False

# Stage 4b's gradient checkpointing is a training-time trade -- recompute activations to save
# memory during the backward pass. There is no backward pass here, so it only costs time.
if hasattr(model.enc, "gradient_checkpointing_disable"):
    model.enc.gradient_checkpointing_disable()

USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

print(f"loaded {MODEL_PATH.name}")
print(f"   attention  {getattr(model.enc.config, '_attn_implementation', ATTN)}")
print(f"   precision  {'bfloat16 autocast' if USE_BF16 else 'float32'}")
print(f"   encoder    frozen, gradient checkpointing off")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

... (2 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

loaded model_attention_2024.pt
   attention  sdpa
   precision  bfloat16 autocast
   encoder    frozen, gradient checkpointing off


In [3]:
import gc

def windows(text, stride=STRIDE):
    ids = tok.encode(text, add_special_tokens=False, truncation=False)
    body = MAXLEN - 2
    if len(ids) <= body:
        return [ids]
    step = body if stride is None else stride
    return [ids[i:i + body] for i in range(0, len(ids), step) if i < len(ids)]


# The probe below measures the real cost; it came back at 13 MB per 512-token sequence with
# SDPA and bf16, down from 63. 700,000 tokens is about 1,370 full windows and roughly 18 GB --
# deliberate headroom, because the halving path is a safety net rather than a plan.
TOKEN_BUDGET = 700_000


def token_batches(wins, budget):
    """
    Indices grouped so each batch's PADDED token count stays under `budget`.

    Longest first. Padding is not only wasted compute -- SDPA's fastest kernel cannot be used
    when an attention mask is present, so grouping similar lengths reduces masking as well.
    The caller's `owner` list is positional, so the original order is restored before returning.
    """
    order = sorted(range(len(wins)), key=lambda i: -len(wins[i]))
    batch, longest = [], 0
    for i in order:
        L = min(MAXLEN, len(wins[i]) + 2)
        nxt = max(longest, L)
        if batch and nxt * (len(batch) + 1) > budget:
            yield batch
            batch, longest = [i], L
        else:
            batch.append(i)
            longest = nxt
    if batch:
        yield batch


@torch.no_grad()
def _encode_batch(idx, wins, out):
    """One forward pass. Separate function so its locals die when it returns OR raises."""
    chunk = [wins[i] for i in idx]
    L = min(MAXLEN, max(len(w) for w in chunk) + 2)
    ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(chunk), L), dtype=torch.long)
    for j, w in enumerate(chunk):
        seq = [tok.cls_token_id] + w[:L - 2] + [tok.sep_token_id]
        ids[j, :len(seq)] = torch.tensor(seq)
        att[j, :len(seq)] = 1
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        h = model.enc(input_ids=ids.to(DEVICE),
                      attention_mask=att.to(DEVICE)).last_hidden_state[:, 0]
    for j, i in enumerate(idx):
        # clone, not a view. h[j] shares storage with the whole batch, so keeping one row would
        # pin all of h -- the entire encoder output -- for as long as `out` lives. The clone
        # copies 3 KB and lets the batch go.
        out[i] = h[j].float().clone()
    del ids, att, h


def _encode(idx, wins, out):
    """
    Encode one batch, halving and retrying if it does not fit.

    THE RETRY MUST HAPPEN OUTSIDE THE except BLOCK.

        An earlier version recursed inside the handler. Python keeps the raising frame alive
        through the traceback for as long as the exception is being handled, so every failed
        attempt pinned its own `ids`, `att` and partial encoder tensors on the GPU. Each retry
        chained another traceback onto the last, and empty_cache() could free none of it because
        all of it was still referenced. Memory climbed until a 20 MB allocation failed -- the
        retry mechanism was the leak.

        Catching, discarding, and only then recursing lets the failed frame be collected first,
        so each retry starts from the memory the previous one released.
    """
    try:
        _encode_batch(idx, wins, out)
        return
    except torch.cuda.OutOfMemoryError:
        pass                        # traceback released when this block exits

    if len(idx) == 1:
        torch.cuda.empty_cache()
        raise RuntimeError(
            f"a single {min(MAXLEN, len(wins[idx[0]]) + 2)}-token window will not fit; "
            f"the GPU is occupied by something other than this batch")

    gc.collect()
    torch.cuda.empty_cache()
    mid = len(idx) // 2
    _encode(idx[:mid], wins, out)
    _encode(idx[mid:], wins, out)


@torch.no_grad()
def encode_windows(wins):
    out = [None] * len(wins)
    for idx in token_batches(wins, TOKEN_BUDGET):
        _encode(idx, wins, out)
    return torch.stack(out)


@torch.no_grad()
def logits_for(pieces):
    """
    Raw logits for one commit: the reduction CommitClassifier.forward performs, with the encoder
    batched. Windows mean-pooled within their piece, pieces attention-pooled into a commit.
    """
    wins, owner = [], []
    for pi, p in enumerate(pieces):
        for w in windows(p):
            wins.append(w)
            owner.append(pi)

    h = encode_windows(wins)

    owner_t = torch.tensor(owner, device=DEVICE)
    uniq, inv = torch.unique(owner_t, return_inverse=True)
    pv = torch.zeros(len(uniq), h.size(1), device=DEVICE, dtype=h.dtype)
    pv.index_add_(0, inv, h)
    cnt = torch.zeros(len(uniq), device=DEVICE, dtype=h.dtype)
    cnt.index_add_(0, inv, torch.ones(len(h), device=DEVICE, dtype=h.dtype))
    pv = pv / cnt.unsqueeze(1)

    a = model.score(pv).softmax(0)
    v = (pv * a).sum(0)
    return model.out(model.drop(v)).float().cpu().numpy()


# --- a measured check before committing to a 34,000-commit run -------------------------------
if DEVICE == "cuda":
    probe = [list(range(MAXLEN - 2))] * 512          # 512 full-length windows
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    _ = encode_windows(probe)
    per_seq = torch.cuda.max_memory_allocated() / len(probe) / 1e6
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"probe: {per_seq:.0f} MB per 512-token sequence "
          f"({'was 63 MB with eager attention in fp32' if per_seq < 50 else 'no saving -- check '
             'that SDPA is active above'})")
    print(f"   {TOKEN_BUDGET:,} tokens = {TOKEN_BUDGET // MAXLEN:,} full windows "
          f"-> ~{TOKEN_BUDGET / MAXLEN * per_seq / 1024:.0f} GB peak")


# Scoring 34,609 commits needs a GPU (hours even on an A100). Without one, the logits this study
# published are downloaded instead and the rest of the notebook runs from them on a CPU.
# CIC_PUBLISHED_LOGITS=1 forces that; =0 scores regardless of hardware.
_published = os.environ.get("CIC_PUBLISHED_LOGITS", "auto")
if _published == "1" or (_published == "auto" and DEVICE != "cuda"):
    require("stage5_logits.jsonl")
    print("using the published stage5_logits.jsonl; classification is not re-run")

LOGITS_PATH = DRIVE / "stage5_logits.jsonl"
done = set()
if LOGITS_PATH.exists():
    for line in open(LOGITS_PATH, encoding="utf-8"):
        if line.strip():
            done.add(json.loads(line)["sha"])
todo = C[~C.sha.isin(done)]
print(f"{len(done):,} already scored; {len(todo):,} to score")

if len(todo):
    todo = todo.assign(_p=todo.pieces.map(len)).sort_values("_p", ascending=False)
    t0, peak = time.time(), 0.0
    recent = collections.deque(maxlen=200)
    with open(LOGITS_PATH, "a", encoding="utf-8") as f:
        for n, (_, row) in enumerate(todo.iterrows(), 1):
            tick = time.time()
            lg = logits_for(row["pieces"])
            f.write(json.dumps({"sha": row["sha"], "logits": [float(x) for x in lg]}) + "\n")
            recent.append(time.time() - tick)
            if n % 100 == 0 or n == len(todo):
                f.flush()
                if DEVICE == "cuda":
                    peak = max(peak, torch.cuda.max_memory_allocated() / 1e9)
                    torch.cuda.reset_peak_memory_stats()
                per = sum(recent) / len(recent)
                print(f"\r   {n:,}/{len(todo):,}   {60/max(1e-9, per):,.0f}/min   "
                      f"~{(len(todo)-n)*per/60:.0f} min left   peak {peak:.0f} GB",
                      end="", flush=True)
    print(f"\n\ndone in {(time.time()-t0)/60:.1f} min")

probe: 13 MB per 512-token sequence (was 63 MB with eager attention in fp32)
   700,000 tokens = 1,367 full windows -> ~17 GB peak
34,609 already scored; 0 to score


## 3. Calibrate

Raw softmax is not a probability — a network saying 0.99 is right rather less often than that.
**Temperature scaling** (Guo et al., ICML 2017) divides the logits by one scalar before the
softmax. It cannot change a prediction, since all three logits move together; it only makes the
confidence mean what it says.

**T = 2.86**, fitted in Stage 4c on Levin's held-out commits, took expected calibration error
from 0.109 to 0.025. More flexible methods — isotonic regression, vector scaling — have many more
parameters and would overfit 173 calibration points.

**Margin** — top class minus runner-up — is the confidence score used below. Stage 4c found it
separates correct from incorrect predictions at least as well as the top probability, and it
distinguishes *confidently corrective* from *torn between corrective and adaptive*.

In [4]:
L = pd.read_json(LOGITS_PATH, lines=True).drop_duplicates(subset="sha", keep="last")
C = C.merge(L, on="sha", how="left")
if C.logits.isna().any():
    raise SystemExit(f"{int(C.logits.isna().sum())} commit(s) unscored -- re-run section 2")

Z = np.stack(C.logits.values)
def softmax(z, T=1.0):
    z = z / T
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

P = softmax(Z, TEMPERATURE)
S = np.sort(P, axis=1)
C["predicted"] = [LABELS[i] for i in P.argmax(1)]
C["confidence"] = S[:, -1]
C["margin"] = S[:, -1] - S[:, -2]

print("PREDICTED CLASS, before any threshold\n")
t = pd.crosstab(C.repo, C.predicted, normalize="index").mul(100).round(1)
t["n"] = C.groupby("repo").size()
print(t.to_string())

print("\n\nCALIBRATED MARGIN\n")
counts, edges = np.histogram(C.margin, bins=20, range=(0, 1))
for lo, n in zip(edges[:-1], counts):
    print(f"   {lo:.2f}-{lo+0.05:.2f}  {n:>7,}  {'#' * int(n / max(counts) * 52)}")

PREDICTED CLASS, before any threshold

predicted        adaptive  corrective  perfective      n
repo                                                    
checkstyle           15.6        22.2        62.2  10240
commons-lang         10.5        14.6        74.9   6818
dubbo                16.8        40.5        42.7   6681
junit-framework      20.3        12.9        66.8   6142
questdb              20.2        41.4        38.4   4728


CALIBRATED MARGIN

   0.00-0.05      895  ####
   0.05-0.10      918  ####
   0.10-0.15      780  ####
   0.15-0.20      756  ####
   0.20-0.25      714  ###
   0.25-0.30      760  ####
   0.30-0.35      731  ###
   0.35-0.40      731  ###
   0.40-0.45      779  ####
   0.45-0.50      827  ####
   0.50-0.55      911  ####
   0.55-0.60      968  #####
   0.60-0.65    1,162  ######
   0.65-0.70    1,249  ######
   0.70-0.75    1,617  ########
   0.75-0.80    2,101  ###########
   0.80-0.85    2,983  ################
   0.85-0.90    5,983  #################

## 4. The boundary

The distribution above has **no knee**: counts are roughly flat across the low half and rise into
a single mass near the top. There is no separate population of unclassifiable commits to find, so
any cut is a judgement — and a judgement needs a stated basis.

**Primary: 0.47, from Levin's inter-rater agreement.** Two trained annotators agreed on 79% of a
110-commit sample before sharing guidelines, so about one commit in five was contestable between
humans. Setting aside a comparable share is a claim about the task, not about this model.

That agreement figure measures *which* class two people chose, not whether a commit was
classifiable, so it is a proxy for the task's ambiguity rather than a direct measurement of it.

**Sensitivity: 0.34, from Stage 4c.** It was calibrated to reach 90% precision on Levin's held-out
commits — but that population was filtered for legibility, since commits the authors could not
classify were dropped and resampled. A threshold fitted there will be too permissive on a corpus
that includes terse and uninformative commits, which is the reason it is secondary here.

Both are reported, so a downstream result can be checked against the choice.

In [5]:
THRESHOLD, SENSITIVITY = 0.47, 0.34

print(f"{'threshold':>10}{'below':>10}{'% of corpus':>13}")
print("-" * 33)
for t in (0.10, 0.20, SENSITIVITY, 0.40, THRESHOLD, 0.60):
    below = int((C.margin < t).sum())
    tag = ("   <- primary" if t == THRESHOLD else "   <- sensitivity" if t == SENSITIVITY else "")
    print(f"{t:>10.2f}{below:>10,}{below/len(C)*100:>12.1f}%{tag}")


def assign(margin_threshold):
    return np.where(C.margin < margin_threshold, "unclassified", C.predicted)


C["swanson_class"] = assign(THRESHOLD)
C["swanson_class_at_034"] = assign(SENSITIVITY)
C["is_low_confidence"] = C.margin < THRESHOLD

for name, col in (("PRIMARY, 0.47", "swanson_class"), ("SENSITIVITY, 0.34", "swanson_class_at_034")):
    print(f"\n\n{name}\n")
    t = pd.crosstab(C.repo, C[col], normalize="index").mul(100).round(1)
    t["n"] = C.groupby("repo").size()
    print(t.to_string())

print("\n\nWHAT CHANGES BETWEEN THE TWO\n")
moved = C.swanson_class != C.swanson_class_at_034
print(f"   {int(moved.sum()):,} commits ({moved.mean()*100:.1f}%) move to unclassified at 0.47")
print(f"   they were: " + ", ".join(f"{k} {v:,}" for k, v in
      C.loc[moved, "swanson_class_at_034"].value_counts().items()))

 threshold     below  % of corpus
---------------------------------
      0.10     1,813         5.2%
      0.20     3,349         9.7%
      0.34     5,390        15.6%   <- sensitivity
      0.40     6,285        18.2%
      0.47     7,395        21.4%   <- primary
      0.60     9,770        28.2%


PRIMARY, 0.47

swanson_class    adaptive  corrective  perfective  unclassified      n
repo                                                                  
checkstyle           11.4        16.4        52.1          20.1  10240
commons-lang          6.7         9.6        64.4          19.2   6818
dubbo                10.8        27.5        30.1          31.6   6681
junit-framework      14.0         7.5        56.4          22.1   6142
questdb              17.3        37.8        33.2          11.8   4728


SENSITIVITY, 0.34

swanson_class_at_034  adaptive  corrective  perfective  unclassified      n
repo                                                                       
checkstyle 

## 5. Does the confidence mean anything on this corpus?

Temperature was fitted on Levin's commits. Whether the confidence it produces is informative
*here* is a separate question, and this corpus offers one way to test it: commits linked to an
issue or pull request whose maintainer label implies a Swanson class.

**The labels are a weak reference.** Herzig, Just & Zeller found about a third of bug-labelled
reports are not bugs, and most labels in this corpus describe process rather than the kind of
work. So agreement below 100% is expected everywhere. What matters is the **shape**:

- if agreement **rises with margin**, confident predictions are more often right and the threshold
  is setting aside the commits it should
- if agreement is **flat**, confidence carries no information on this corpus, and the fourth class
  is an arbitrary slice rather than a meaningful one

Only commits whose labels imply exactly one class are used. Those implying several are ambiguous
and excluded, not resolved.

In [6]:
REF = C[C.label_class.isin(LABELS)].copy()
REF["agree"] = REF.predicted == REF.label_class
print(f"{len(REF):,} commits carry a label implying exactly one class "
      f"({len(REF)/len(C)*100:.1f}% of the corpus)\n")
print(REF.groupby("repo").size().rename("reference commits").to_string())

if len(REF) < 100:
    print("\nToo few labelled commits for a meaningful check.")
else:
    print(f"\n\noverall agreement with maintainer labels: {REF.agree.mean()*100:.1f}%\n")

    # Agreement by margin decile. Deciles rather than fixed bins, so every bin holds enough
    # commits to be read -- fixed bins would leave the low end nearly empty.
    REF["margin_bin"] = pd.qcut(REF.margin, q=10, duplicates="drop")
    by_bin = REF.groupby("margin_bin", observed=True).agg(
        n=("agree", "size"), agreement=("agree", "mean"),
        margin_lo=("margin", "min"), margin_hi=("margin", "max"))
    by_bin["agreement"] = (by_bin.agreement * 100).round(1)
    print("AGREEMENT BY CALIBRATED MARGIN\n")
    for _, r in by_bin.iterrows():
        print(f"   {r.margin_lo:.2f}-{r.margin_hi:.2f}  n={int(r.n):>5,}  "
              f"{r.agreement:>5.1f}%  {'#' * int(r.agreement / 2)}")

    from scipy.stats import spearmanr
    rho, p = spearmanr(REF.margin, REF.agree.astype(int))
    lo, hi = by_bin.agreement.iloc[0], by_bin.agreement.iloc[-1]
    print(f"\n   lowest decile {lo:.1f}%   highest decile {hi:.1f}%   "
          f"spread {hi - lo:+.1f} points")
    print(f"   Spearman rho {rho:.3f}  (p = {p:.2g})")

    print("\n\nAGREEMENT EITHER SIDE OF EACH THRESHOLD\n")
    for t in (SENSITIVITY, THRESHOLD):
        above, below = REF[REF.margin >= t], REF[REF.margin < t]
        print(f"   {t:.2f}   above {above.agree.mean()*100:5.1f}% (n={len(above):,})   "
              f"below {below.agree.mean()*100:5.1f}% (n={len(below):,})   "
              f"gap {(above.agree.mean()-below.agree.mean())*100:+.1f}")

    print("\n\nBY REPOSITORY\n")
    per = REF.groupby("repo").agg(n=("agree", "size"), agreement=("agree", "mean"))
    per["agreement"] = (per.agreement * 100).round(1)
    print(per.to_string())

    print("\n\nCONFUSION WITH MAINTAINER LABELS\n")
    print(pd.crosstab(REF.label_class, REF.predicted, margins=True).to_string())

    rising = rho > 0.1 and p < 0.01 and (hi - lo) > 10
    print("\n" + "=" * 66)
    if rising:
        print("Agreement rises with margin. The calibrated confidence is informative on this")
        print("corpus, and commits below the threshold are the ones the model is least reliable on.")
    else:
        print("Agreement does not clearly rise with margin. On this corpus the confidence carries")
        print("little information, so the fourth class is a slice by score rather than a set of")
        print("commits the model is demonstrably less reliable on. Report it that way.")
    print("=" * 66)

4,422 commits carry a label implying exactly one class (12.8% of the corpus)

repo
checkstyle         1181
dubbo               339
junit-framework    1647
questdb            1255


overall agreement with maintainer labels: 58.3%

AGREEMENT BY CALIBRATED MARGIN

   0.00-0.22  n=  443   37.2%  ##################
   0.22-0.43  n=  442   43.7%  #####################
   0.43-0.60  n=  442   50.0%  #########################
   0.60-0.73  n=  442   50.0%  #########################
   0.73-0.81  n=  442   58.8%  #############################
   0.81-0.86  n=  442   69.2%  ##################################
   0.86-0.89  n=  442   63.8%  ###############################
   0.89-0.91  n=  442   72.6%  ####################################
   0.91-0.92  n=  442   78.3%  #######################################
   0.92-0.95  n=  443   59.1%  #############################

   lowest decile 37.2%   highest decile 59.1%   spread +21.9 points
   Spearman rho 0.212  (p = 4.1e-46)


AGREEMENT EITHER SIDE O

## 6. Write the dataset

In [7]:
COLUMNS = ["repo", "sha", "timestamp", "release_interval",
           "swanson_class", "swanson_class_at_034", "predicted", "confidence", "margin",
           "is_low_confidence",
           "has_refactoring", "n_refactorings", "n_high", "n_medium", "n_low",
           "refactoring_types",
           "has_evidence", "evidence_routes", "maintainer_labels", "label_class",
           "n_files", "n_pieces_uncapped"]

OUT = Path("stage5"); OUT.mkdir(exist_ok=True)
A = C[COLUMNS].copy()
for c in ("confidence", "margin"):
    A[c] = A[c].round(4)
A.to_csv(OUT / "dataset.csv", index=False)
A.to_csv(DRIVE / "dataset.csv", index=False)

meta = {
    "commits": int(len(A)),
    "model": MODEL_PATH.name,
    "temperature": TEMPERATURE,
    "temperature_source": "Stage 4c, fitted on Levin's held-out commits; ECE 0.109 -> 0.025",
    "confidence_score": "margin: top calibrated probability minus the runner-up",
    "threshold": THRESHOLD,
    "threshold_source": ("Levin & Yehudai's pre-guideline inter-rater agreement of 79%; the corpus "
                         "margin distribution showed no knee, so the cut is a stated judgement "
                         "rather than a discovered boundary"),
    "sensitivity_threshold": SENSITIVITY,
    "sensitivity_source": "Stage 4c, 90% precision on Levin's legibility-filtered test set",
    "class_distribution": {k: int(v) for k, v in A.swanson_class.value_counts().items()},
    "class_distribution_at_034": {k: int(v) for k, v in
                                  A.swanson_class_at_034.value_counts().items()},
    "reference_commits": int(len(REF)),
    "with_refactoring": int(A.has_refactoring.sum()),
    "with_evidence": int(A.has_evidence.sum()),
    "with_interval": int((A.release_interval.astype(str) != "").sum()),
}
json.dump(meta, open(DRIVE / "dataset_meta.json", "w"), indent=1)

print(f"dataset.csv   {len(A):,} rows x {len(COLUMNS)} columns   (local and Drive)\n")
print(json.dumps(meta, indent=1))

dataset_a.csv   34,609 rows x 22 columns   (local and Drive)

{
 "commits": 34609,
 "model": "model_attention_2024.pt",
 "temperature": 2.86,
 "temperature_source": "Stage 4c, fitted on Levin's held-out commits; ECE 0.109 -> 0.025",
 "confidence_score": "margin: top calibrated probability minus the runner-up",
 "threshold": 0.47,
 "threshold_source": "Levin & Yehudai's pre-guideline inter-rater agreement of 79%; the corpus margin distribution showed no knee, so the cut is a stated judgement rather than a discovered boundary",
 "sensitivity_threshold": 0.34,
 "sensitivity_source": "Stage 4c, 90% precision on Levin's legibility-filtered test set",
 "class_distribution": {
  "perfective": 16769,
  "unclassified": 7395,
  "corrective": 6416,
  "adaptive": 4029
 },
 "class_distribution_at_034": {
  "perfective": 17757,
  "corrective": 6980,
  "unclassified": 5390,
  "adaptive": 4482
 },
 "reference_commits": 4422,
 "with_refactoring": 15752,
 "with_evidence": 19211,
 "with_interval": 30042


In [8]:
import pandas as pd
A = pd.read_csv("stage5/dataset.csv")
print(pd.crosstab(A.has_refactoring, A.predicted, normalize="index").mul(100).round(1))

predicted        adaptive  corrective  perfective
has_refactoring                                  
False                14.2        25.7        60.1
True                 18.8        24.6        56.5
